### Курсовая работа Вариант 12
### Ньяти Каелиле БВТ2201

#### Задание №12.

Вычислительная система (ВС) состоит из 3-х серверов, обрабатывающих программы. 

Программы поступают случайным образом, распределенные по линейному закону: Tzmin=1/3 сек, Tzmax=2/3 сек. Если 1-ый сервер занят, то программы обрабатываются 2-ым сервером и т.д. Если и 1-ый и 2-ой и 3-ий серверы заняты, то программа покидает ВС необработанной.

Время обработки одной программы каждым сервером – случайная величина, распределенная по линейному закону: Tsmin=1 сек, Tsmax=6 сек. Разработать программу, моделирующую работу ВС и найти ее характеристики за время работы 1 час. Характеристики ВС: 
    
    • P0 – вероятность того, что ВС не загружена, 
    
    • P1 – вероятность того, что загружен только один сервер, 
    
    • P2 – вероятность того, что загружены два сервера, 
    
    • P3 – вероятность того, что загружены три сервера
    
    • Q – относительная пропускная способность ВС – средняя доля программ, обработанных ВС,
    
    • S – абсолютная пропускная способность – среднее число программ, обработанных в единицу времени,
    
    • Pотк – вероятность отказа, т.е. того, что программа будет не обработанной,
    
    • K - среднее число занятых серверов.

Найти характеристики ВС, если программы поступают случайным образом, распределенные по экспоненциальному закону с частотой λ=2 1/сек, а среднее время обработки программы каждым сервером составляет tобр= 3 сек (закон распределения -экспоненциальный).


### 1. Импорт библиотек и настройка

In [11]:
# Импорт необходимых библиотек
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
np.random.seed(42)  # Для воспроизводимости результатов

print("Библиотеки успешно импортированы!")
print(f"Версия NumPy: {np.__version__}")

Библиотеки успешно импортированы!
Версия NumPy: 1.22.0


### 2. Класс для имитационной модели ВС

In [12]:
class ComputingSystemSimulator:
    """
    Класс для имитационного моделирования вычислительной системы
    с 3 серверами без очереди
    """
    
    def __init__(self, lambda_arrival=None, t_obr_mean=None, 
                 tz_min=None, tz_max=None, ts_min=None, ts_max=None,
                 sim_time=3600, distribution_type='exponential'):
        """
        Инициализация параметров модели
        
        Параметры:
        - lambda_arrival: интенсивность входного потока (для экспоненциального)
        - t_obr_mean: среднее время обслуживания (для экспоненциального)
        - tz_min, tz_max: мин и макс интервалы поступления (для линейного)
        - ts_min, ts_max: мин и макс время обслуживания (для линейного)
        - sim_time: время моделирования (сек)
        - distribution_type: тип распределения ('exponential' или 'linear')
        """
        self.sim_time = sim_time
        self.distribution_type = distribution_type
        self.num_servers = 3
        
        # Параметры для экспоненциального распределения (Режим Б)
        self.lambda_arrival = lambda_arrival
        self.t_obr_mean = t_obr_mean
        
        # Параметры для линейного распределения (Режим А)
        self.tz_min = tz_min
        self.tz_max = tz_max
        self.ts_min = ts_min
        self.ts_max = ts_max
        
        # Счетчики для сбора статистики
        self.reset_statistics()
        
    def reset_statistics(self):
        """Сброс всех статистических счетчиков"""
        # Состояния серверов (0 - свободен, 1 - занят)
        self.servers = [0, 0, 0]
        
        # Времена освобождения серверов
        self.server_free_times = [0, 0, 0]
        
        # Счетчики
        self.total_requests = 0
        self.processed_requests = 0
        self.rejected_requests = 0
        
        # Время нахождения в каждом состоянии
        self.state_time = [0, 0, 0, 0]  # P0, P1, P2, P3
        self.last_event_time = 0
        
        # Текущее состояние (количество занятых серверов)
        self.current_state = 0
        
    def generate_interarrival_time(self):
        """Генерация интервала между поступлениями"""
        if self.distribution_type == 'exponential':
            # Экспоненциальное распределение
            u = random.random()
            return -(1/self.lambda_arrival) * np.log(u)
        else:
            # Линейное (равномерное) распределение
            return (self.tz_max - self.tz_min) * random.random() + self.tz_min
        
    
    def generate_service_time(self):
        """Генерация времени обслуживания"""
        if self.distribution_type == 'exponential':
            # Экспоненциальное распределение
            u = random.random()
            return -(self.t_obr_mean) * np.log(u)
        else:
            # Линейное (равномерное) распределение
            return (self.ts_max - self.ts_min) * random.random() + self.ts_min
    
    def get_state(self):
        """Получение текущего состояния (количество занятых серверов)"""
        return sum(self.servers)
    
    def update_state_time(self, current_time):
        """Обновление времени пребывания в состоянии"""
        time_delta = current_time - self.last_event_time
        if time_delta > 0:
            self.state_time[self.current_state] += time_delta
        self.last_event_time = current_time
    
    def find_free_server(self):
        """Поиск свободного сервера (последовательный перебор)"""
        for i in range(self.num_servers):
            if self.servers[i] == 0:
                return i
        return -1
    
    def run_simulation(self):
        """Запуск имитационного моделирования"""
        self.reset_statistics()
        
        # Генерация первого события
        current_time = self.generate_interarrival_time()
        
        # Список событий (время, тип, сервер)
        # Типы событий: 'arrival' - приход, 'departure' - уход
        events = [(current_time, 'arrival', -1)]
        
        while events and current_time <= self.sim_time:
            # Сортировка событий по времени
            events.sort(key=lambda x: x[0])
            
            # Получение следующего события
            current_time, event_type, server_id = events.pop(0)
            
            if current_time > self.sim_time:
                break
            
            # Обновление времени в состоянии
            self.update_state_time(current_time)
            
            if event_type == 'arrival':
                # Обработка прихода заявки
                self.total_requests += 1
                
                # Поиск свободного сервера
                free_server = self.find_free_server()
                
                if free_server != -1:
                    # Заявка принята на обслуживание
                    self.processed_requests += 1
                    
                    # Занимаем сервер
                    self.servers[free_server] = 1
                    
                    # Генерируем время обслуживания
                    service_time = self.generate_service_time()
                    departure_time = current_time + service_time
                    
                    # Добавляем событие окончания обслуживания
                    events.append((departure_time, 'departure', free_server))
                    
                    # Обновляем время освобождения сервера
                    self.server_free_times[free_server] = departure_time
                else:
                    # Все серверы заняты - отказ
                    self.rejected_requests += 1
                
                # Планируем следующее поступление
                next_arrival = current_time + self.generate_interarrival_time()
                if next_arrival <= self.sim_time:
                    events.append((next_arrival, 'arrival', -1))
                    
            elif event_type == 'departure':
                # Обработка ухода заявки
                self.servers[server_id] = 0
                self.server_free_times[server_id] = 0
            
            # Обновляем текущее состояние
            self.current_state = self.get_state()
        
        # Добавляем время до конца моделирования
        if self.last_event_time < self.sim_time:
            self.state_time[self.current_state] += (self.sim_time - self.last_event_time)
        
        # Расчет характеристик
        results = self.calculate_characteristics()
        return results
    
    def calculate_characteristics(self):
        """Расчет характеристик ВС"""
        total_time = self.sim_time
        
        # Вероятности состояний
        P = [t / total_time for t in self.state_time[:4]]
        
        # Дополняем до 4-х элементов, если нужно
        while len(P) < 4:
            P.append(0)
        
        # Относительная пропускная способность
        Q = self.processed_requests / self.total_requests if self.total_requests > 0 else 0
        
        # Абсолютная пропускная способность (заявок в секунду)
        S = self.processed_requests / self.sim_time
        
        # Вероятность отказа
        P_reject = self.rejected_requests / self.total_requests if self.total_requests > 0 else 0
        
        # Среднее число занятых серверов
        K = sum(i * P[i] for i in range(4))
        
        results = {
            'P0': P[0],
            'P1': P[1],
            'P2': P[2],
            'P3': P[3],
            'Q': Q,
            'S': S,
            'P_reject': P_reject,
            'K': K,
            'total_requests': self.total_requests,
            'processed': self.processed_requests,
            'rejected': self.rejected_requests
        }
        
        return results
    
    def run_multiple_simulations(self, num_runs=5):
        """Запуск нескольких прогонов модели"""
        all_results = []
        
        for run in range(num_runs):
            # Разные seed для разных прогонов
            np.random.seed(42 + run)
            results = self.run_simulation()
            all_results.append(results)
        
        return all_results

print("Класс ComputingSystemSimulator успешно создан!")

Класс ComputingSystemSimulator успешно создан!


### 3. Аналитические расчеты для верификации (Экспоненциальный закон)

In [13]:
# Параметры для режима Б (экспоненциальный)
lambda_arrival = 2  # заявок/сек
t_obr_mean = 3  # сек
mu = 1 / t_obr_mean  # интенсивность обслуживания

# Интенсивность нагрузки
rho = lambda_arrival / mu
print("=" * 60)
print("АНАЛИТИЧЕСКИЙ РАСЧЕТ ДЛЯ ЭКСПОНЕНЦИАЛЬНОГО ЗАКОНА (M/M/3)")
print("=" * 60)
print(f"Интенсивность входного потока (λ): {lambda_arrival} заявок/сек")
print(f"Среднее время обслуживания (t_обр): {t_obr_mean} сек")
print(f"Интенсивность обслуживания (μ): {mu:.3f} заявок/сек")
print(f"Количество серверов (n): 3")
print(f"Приведенная интенсивность нагрузки (ρ = λ/μ): {rho:.2f}")
print()

# Расчет предельных вероятностей по формулам Эрланга
sum_rho = 0
for k in range(4):  # 0, 1, 2, 3
    sum_rho += (rho**k) / np.math.factorial(k)

P0 = 1 / sum_rho
P1 = (rho**1 / np.math.factorial(1)) * P0
P2 = (rho**2 / np.math.factorial(2)) * P0
P3 = (rho**3 / np.math.factorial(3)) * P0

# Вероятность отказа = P3
P_reject = P3

# Относительная пропускная способность
Q = 1 - P_reject

# Абсолютная пропускная способность
S = lambda_arrival * Q

# Среднее число занятых серверов
K = rho * Q

print("ПРЕДЕЛЬНЫЕ ВЕРОЯТНОСТИ СОСТОЯНИЙ:")
print(f"P0 (все серверы свободны) = {P0:.6f} ({P0*100:.2f}%)")
print(f"P1 (занят 1 сервер)       = {P1:.6f} ({P1*100:.2f}%)")
print(f"P2 (заняты 2 сервера)     = {P2:.6f} ({P2*100:.2f}%)")
print(f"P3 (заняты 3 сервера)     = {P3:.6f} ({P3*100:.2f}%)")
print()
print("ХАРАКТЕРИСТИКИ ЭФФЕКТИВНОСТИ:")
print(f"Вероятность отказа (Pотк): {P_reject:.6f} ({P_reject*100:.2f}%)")
print(f"Относительная пропускная способность (Q): {Q:.6f} ({Q*100:.2f}%)")
print(f"Абсолютная пропускная способность (S): {S:.6f} заявок/сек")
print(f"Среднее число занятых серверов (K): {K:.6f}")

# Сохраняем для сравнения
analytical_results = {
    'P0': P0, 'P1': P1, 'P2': P2, 'P3': P3,
    'Q': Q, 'S': S, 'P_reject': P_reject, 'K': K
}

АНАЛИТИЧЕСКИЙ РАСЧЕТ ДЛЯ ЭКСПОНЕНЦИАЛЬНОГО ЗАКОНА (M/M/3)
Интенсивность входного потока (λ): 2 заявок/сек
Среднее время обслуживания (t_обр): 3 сек
Интенсивность обслуживания (μ): 0.333 заявок/сек
Количество серверов (n): 3
Приведенная интенсивность нагрузки (ρ = λ/μ): 6.00

ПРЕДЕЛЬНЫЕ ВЕРОЯТНОСТИ СОСТОЯНИЙ:
P0 (все серверы свободны) = 0.016393 (1.64%)
P1 (занят 1 сервер)       = 0.098361 (9.84%)
P2 (заняты 2 сервера)     = 0.295082 (29.51%)
P3 (заняты 3 сервера)     = 0.590164 (59.02%)

ХАРАКТЕРИСТИКИ ЭФФЕКТИВНОСТИ:
Вероятность отказа (Pотк): 0.590164 (59.02%)
Относительная пропускная способность (Q): 0.409836 (40.98%)
Абсолютная пропускная способность (S): 0.819672 заявок/сек
Среднее число занятых серверов (K): 2.459016


### 4. Верификация - 5 прогонов для экспоненциального закона

In [14]:
print("=" * 60)
print("ВЕРИФИКАЦИЯ: 5 ПРОГОНОВ МОДЕЛИ (ЭКСПОНЕНЦИАЛЬНЫЙ ЗАКОН)")
print("=" * 60)

# Создаем симулятор для экспоненциального закона
sim_exp = ComputingSystemSimulator(
    lambda_arrival=lambda_arrival,
    t_obr_mean=t_obr_mean,
    sim_time=3600,  # 1 час
    distribution_type='exponential'
)

# Запускаем 5 прогонов
exp_results = sim_exp.run_multiple_simulations(num_runs=5)

# Создаем DataFrame для результатов
df_exp = pd.DataFrame(exp_results)

# Добавляем столбец с номером запуска
df_exp.insert(0, 'Run', range(1, 6))

# Округляем для красивого отображения
df_exp_display = df_exp[['Run', 'P0', 'P1', 'P2', 'P3', 'Q', 'S', 'K']].copy()
for col in ['P0', 'P1', 'P2', 'P3', 'Q', 'S', 'K']:
    df_exp_display[col] = df_exp_display[col].apply(lambda x: f"{x:.6f}")

print("Таблица 1. Результаты работы программы при пяти запусках (Экспоненциальный закон)")
print(df_exp_display.to_string(index=False))
print()

# Рассчитываем средние значения
exp_mean = df_exp[['P0', 'P1', 'P2', 'P3', 'Q', 'S', 'K']].mean()

# Создаем таблицу сравнения
comparison = pd.DataFrame({
    'Характеристика': ['P0', 'P1', 'P2', 'P3', 'Q', 'S', 'K'],
    'Аналитическое': [analytical_results['P0'], analytical_results['P1'], 
                      analytical_results['P2'], analytical_results['P3'],
                      analytical_results['Q'], analytical_results['S'], 
                      analytical_results['K']],
    'Среднее по модели': [exp_mean['P0'], exp_mean['P1'], exp_mean['P2'], 
                          exp_mean['P3'], exp_mean['Q'], exp_mean['S'], 
                          exp_mean['K']]
})

# Расчет абсолютного и относительного отклонения
comparison['Абс. отклонение'] = comparison['Среднее по модели'] - comparison['Аналитическое']
comparison['Отн. отклонение (%)'] = (comparison['Абс. отклонение'].abs() / comparison['Аналитическое'] * 100)

print("СРАВНЕНИЕ С АНАЛИТИЧЕСКИМИ РАСЧЕТАМИ:")
print(comparison.to_string(index=False, float_format="{:.6f}".format))
print()

print("ВЫВОД:")
print("Полученные отклонения незначительны (менее 1-2%), что подтверждает")
print("корректность работы имитационной модели для экспоненциального случая.")

ВЕРИФИКАЦИЯ: 5 ПРОГОНОВ МОДЕЛИ (ЭКСПОНЕНЦИАЛЬНЫЙ ЗАКОН)
Таблица 1. Результаты работы программы при пяти запусках (Экспоненциальный закон)
 Run       P0       P1       P2       P3        Q        S        K
   1 0.018254 0.097039 0.301704 0.583003 0.419599 0.831389 2.449456
   2 0.011025 0.091262 0.293395 0.604318 0.404726 0.813611 2.491004
   3 0.016119 0.092204 0.296065 0.595612 0.401137 0.803611 2.471169
   4 0.016041 0.108244 0.295915 0.579800 0.421452 0.833889 2.439474
   5 0.020486 0.101643 0.293489 0.584382 0.406084 0.804722 2.441767

СРАВНЕНИЕ С АНАЛИТИЧЕСКИМИ РАСЧЕТАМИ:
Характеристика  Аналитическое  Среднее по модели  Абс. отклонение  Отн. отклонение (%)
            P0       0.016393           0.016385        -0.000008             0.051089
            P1       0.098361           0.098079        -0.000282             0.286766
            P2       0.295082           0.296113         0.001032             0.349570
            P3       0.590164           0.589423        -0.000741  

### 5. Верификация - 10 прогонов для линейного закона

In [15]:
print("=" * 60)
print("ВЕРИФИКАЦИЯ ДЛЯ ЛИНЕЙНОГО ЗАКОНА РАСПРЕДЕЛЕНИЯ")
print("=" * 60)

# Parameters for linear mode (Mode A from your assignment)
tz_min = 1/3      # 0.333 sec
tz_max = 2/3      # 0.667 sec
ts_min = 1        # 1 sec
ts_max = 6        # 6 sec
sim_time = 3600   # 1 hour
num_servers = 3   # 3 servers

print(f"Параметры линейного режима:")
print(f"  Tz ~ Uniform({tz_min:.3f}, {tz_max:.3f}) сек")
print(f"  Ts ~ Uniform({ts_min}, {ts_max}) сек")
print(f"  Время моделирования: {sim_time} сек")
print(f"  Количество серверов: {num_servers}")
print()

ВЕРИФИКАЦИЯ ДЛЯ ЛИНЕЙНОГО ЗАКОНА РАСПРЕДЕЛЕНИЯ
Параметры линейного режима:
  Tz ~ Uniform(0.333, 0.667) сек
  Ts ~ Uniform(1, 6) сек
  Время моделирования: 3600 сек
  Количество серверов: 3



In [16]:
sim_test = ComputingSystemSimulator(
    tz_min=tz_min,
    tz_max=tz_max,
    ts_min=ts_min,
    ts_max=ts_max,
    sim_time=sim_time,
    distribution_type='linear'
)

sim_working = ComputingSystemSimulator(
    tz_min=tz_min,
    tz_max=tz_max,
    ts_min=ts_min,
    ts_max=ts_max,
    sim_time=sim_time,
    distribution_type='linear'
)

In [17]:
# Run 10 simulations for the "test" program
test_results = []
for run in range(10):
    np.random.seed(100 + run)  # Different seed for test program
    results = sim_test.run_simulation()
    test_results.append(results)

# Run 10 simulations for the "working" program  
working_results = []
for run in range(10):
    np.random.seed(200 + run)  # Different seed for working program
    results = sim_working.run_simulation()
    working_results.append(results)

# Convert to DataFrames
df_test = pd.DataFrame(test_results)
df_working = pd.DataFrame(working_results)

# Add run numbers
df_test.insert(0, 'Run', range(1, 11))
df_working.insert(0, 'Run', range(1, 11))

In [ ]:
print("=" * 100)
print("Таблица 2. Сопоставление результатов для линейного распределения")
print("=" * 100)

# Create a combined DataFrame for display
combined_data = []
for i in range(10):
    row = {
        'Run_test': i+1,
        'P0_test': f"{df_test.iloc[i]['P0']:.6f}",
        'P1_test': f"{df_test.iloc[i]['P1']:.6f}",
        'P2_test': f"{df_test.iloc[i]['P2']:.6f}",
        'P3_test': f"{df_test.iloc[i]['P3']:.6f}",
        'Run_work': i+1,
        'P0_work': f"{df_working.iloc[i]['P0']:.6f}",
        'P1_work': f"{df_working.iloc[i]['P1']:.6f}",
        'P2_work': f"{df_working.iloc[i]['P2']:.6f}",
        'P3_work': f"{df_working.iloc[i]['P3']:.6f}"
    }
    combined_data.append(row)

# Add statistics rows
combined_data.append({
    'Run_test': 'm',
    'P0_test': f"{df_test['P0'].mean():.6f}",
    'P1_test': f"{df_test['P1'].mean():.6f}",
    'P2_test': f"{df_test['P2'].mean():.6f}",
    'P3_test': f"{df_test['P3'].mean():.6f}",
    'Run_work': 'm',
    'P0_work': f"{df_working['P0'].mean():.6f}",
    'P1_work': f"{df_working['P1'].mean():.6f}",
    'P2_work': f"{df_working['P2'].mean():.6f}",
    'P3_work': f"{df_working['P3'].mean():.6f}"
})

combined_data.append({
    'Run_test': 'S',
    'P0_test': f"{df_test['P0'].std():.6f}",
    'P1_test': f"{df_test['P1'].std():.6f}",
    'P2_test': f"{df_test['P2'].std():.6f}",
    'P3_test': f"{df_test['P3'].std():.6f}",
    'Run_work': 'S',
    'P0_work': f"{df_working['P0'].std():.6f}",
    'P1_work': f"{df_working['P1'].std():.6f}",
    'P2_work': f"{df_working['P2'].std():.6f}",
    'P3_work': f"{df_working['P3'].std():.6f}"
})

# Create DataFrame and display
df_combined = pd.DataFrame(combined_data)

# Create multi-level column headers
print("\n" + " " * 10 + "Результаты тестовых расчетов" + " " * 25 + "Результаты рабочих расчетов")
print("-" * 100)
print(f"{'Run':<8} {'P0':<12} {'P1':<12} {'P2':<12} {'P3':<12} {'Run':<8} {'P0':<12} {'P1':<12} {'P2':<12} {'P3':<12}")
print("-" * 100)

for idx, row in df_combined.iterrows():
    print(f"{row['Run_test']:<8} {row['P0_test']:<12} {row['P1_test']:<12} {row['P2_test']:<12} {row['P3_test']:<12} "
          f"{row['Run_work']:<8} {row['P0_work']:<12} {row['P1_work']:<12} {row['P2_work']:<12} {row['P3_work']:<12}")

Таблица 2. Сопоставление результатов для линейного распределения

          Результаты тестовых расчетов                         Результаты рабочих расчетов
----------------------------------------------------------------------------------------------------
Run      P0           P1           P2           P3           Run      P0           P1           P2           P3          
----------------------------------------------------------------------------------------------------
1        0.000800     0.023132     0.211923     0.764144     1        0.001087     0.023697     0.214847     0.760368    
2        0.001228     0.022457     0.214779     0.761535     2        0.000621     0.022577     0.215554     0.761248    
3        0.000994     0.024220     0.215863     0.758923     3        0.000837     0.023054     0.212519     0.763590    
4        0.001061     0.024623     0.214294     0.760023     4        0.000879     0.028286     0.213659     0.757176    
5        0.001024     0.022819 

### 6. Исследование характеристик для обоих режимов

In [20]:
print("=" * 60)
print("ИССЛЕДОВАНИЕ ХАРАКТЕРИСТИК ВС ДЛЯ ОБОИХ РЕЖИМОВ")
print("=" * 60)

# Функция для запуска эксперимента
def run_experiment(simulator, num_runs=10):
    results = simulator.run_multiple_simulations(num_runs)
    df = pd.DataFrame(results)
    return df.mean()

# Запускаем для режима А (линейный)
print("Режим А (Линейный закон):")
print(f"  Tz ~ Uniform({tz_min}, {tz_max}) сек")
print(f"  Ts ~ Uniform({ts_min}, {ts_max}) сек")
print("-" * 40)

mean_A = run_experiment(sim_working, num_runs=10)
for key in ['P0', 'P1', 'P2', 'P3', 'Q', 'S', 'P_reject', 'K']:
    print(f"  {key}: {mean_A[key]:.6f}")

print()
print("Режим Б (Экспоненциальный закон):")
print(f"  λ = {lambda_arrival} заявок/сек")
print(f"  t_обр = {t_obr_mean} сек")
print("-" * 40)

sim_exp2 = ComputingSystemSimulator(
    lambda_arrival=lambda_arrival,
    t_obr_mean=t_obr_mean,
    sim_time=3600,  # 1 час
    distribution_type='exponential'
)

mean_B = run_experiment(sim_exp2, num_runs=10)
for key in ['P0', 'P1', 'P2', 'P3', 'Q', 'S', 'P_reject', 'K']:
    print(f"  {key}: {mean_B[key]:.6f}")

print()
print("=" * 60)
print("СРАВНИТЕЛЬНЫЙ АНАЛИЗ:")
print("=" * 60)

comparison_final = pd.DataFrame({
    'Характеристика': ['P0', 'P1', 'P2', 'P3', 'Q', 'S', 'P_reject', 'K'],
    'Режим А (Линейный)': [mean_A['P0'], mean_A['P1'], mean_A['P2'], mean_A['P3'], 
                           mean_A['Q'], mean_A['S'], mean_A['P_reject'], mean_A['K']],
    'Режим Б (Экспонента)': [mean_B['P0'], mean_B['P1'], mean_B['P2'], mean_B['P3'], 
                             mean_B['Q'], mean_B['S'], mean_B['P_reject'], mean_B['K']]
})

print(comparison_final.to_string(index=False, float_format="{:.6f}".format))

ИССЛЕДОВАНИЕ ХАРАКТЕРИСТИК ВС ДЛЯ ОБОИХ РЕЖИМОВ
Режим А (Линейный закон):
  Tz ~ Uniform(0.3333333333333333, 0.6666666666666666) сек
  Ts ~ Uniform(1, 6) сек
----------------------------------------
  P0: 0.000844
  P1: 0.023266
  P2: 0.215079
  P3: 0.760812
  Q: 0.391357
  S: 0.782778
  P_reject: 0.608643
  K: 2.735858

Режим Б (Экспоненциальный закон):
  λ = 2 заявок/сек
  t_обр = 3 сек
----------------------------------------
  P0: 0.017563
  P1: 0.096788
  P2: 0.293328
  P3: 0.592321
  Q: 0.410046
  S: 0.819861
  P_reject: 0.589954
  K: 2.460408

СРАВНИТЕЛЬНЫЙ АНАЛИЗ:
Характеристика  Режим А (Линейный)  Режим Б (Экспонента)
            P0            0.000844              0.017563
            P1            0.023266              0.096788
            P2            0.215079              0.293328
            P3            0.760812              0.592321
             Q            0.391357              0.410046
             S            0.782778              0.819861
      P_reject        